In [4]:
import os
import mne
from mne import events_from_annotations, create_info, EpochsArray, concatenate_epochs, Epochs
import json
import pandas as pd
from pathlib import Path
import re
import matplotlib.pyplot as plt
import numpy as np
import warnings
from collections import defaultdict
from eegkit.models import (
    TaskDTO, FilterParamsDTO, TimeDomainParamsDTO, PSDParamsDTO,
    EpochParamsDTO, EpochFullParamsDTO, TableInfoDTO, EpochPSDParamsDTO
)
from notebook_utils import reload_classes, reload_data_classes

warnings.filterwarnings("ignore", message=".*boundary.*data discontinuities.*")
warnings.filterwarnings("ignore", message="FigureCanvasAgg is non-interactive, and thus cannot be shown")

In [5]:
from __future__ import annotations
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple, Union
import pandas as pd


# ---------- 1) Loader (debug-friendly) ----------

def get_participants_df(
    data_root: Union[str, Path],
    release: Union[str, int],
    *,
    normalize_cols: bool = True
) -> pd.DataFrame:
    """
    Load the participants.tsv for a given HBN-EEG FAIR release.

    Parameters
    ----------
    data_root : str | Path
        Root directory that contains cmi_bids_R{release}.
    release : str | int
        Release identifier, e.g. 9 or "R9".
    normalize_cols : bool
        If True, lowercases/strips column names.

    Returns
    -------
    pd.DataFrame
        Participants table as-is (or with normalized column names).
    """
    data_root = Path(data_root)
    rel_str = str(release)
    if not rel_str.startswith("R"):
        rel_str = f"R{rel_str}"
    bids_dir = data_root / f"cmi_bids_{rel_str}"
    pfile = bids_dir / "participants.tsv"

    if not pfile.exists():
        raise FileNotFoundError(f"participants.tsv not found at: {pfile}")

    df = pd.read_csv(pfile, sep="\t")
    if normalize_cols:
        df.columns = [c.strip().lower() for c in df.columns]
    return df


# ---------- 2) Filter (uses loader above) ----------

def filter_subjects(
    data_root: Union[str, Path],
    release: Union[str, int],
    *,
    age_range: Optional[Tuple[float, float]] = None,          # e.g. (5, 15)
    sex: Optional[Iterable[str]] = None,                      # e.g. {"M","F"} or {"F"}
    handedness: Optional[Iterable[str]] = None,               # e.g. {"R","L","A"}
    factor_ranges: Optional[Dict[str, Tuple[float, float]]] = None,  # {"p_factor":(0,2), ...}
    include_nan: bool = False,                                # keep NaNs when filtering
    limit: Optional[int] = None,                              # cap the number of subjects returned
    participants_df: Optional[pd.DataFrame] = None            # pass your own df to avoid reloading (debug)
) -> List[str]:
    """
    Return BIDS subject IDs ('sub-...') from HBN-EEG participants that match filters.

    You can pass a preloaded `participants_df` (e.g., from get_participants_df) to
    debug/inspect before filtering.
    """
    # Load (or reuse) participants
    df = participants_df if participants_df is not None else get_participants_df(data_root, release)
    cols = set(df.columns)

    # Identify subject id column
    id_col = next((c for c in ("participant_id", "subject_id", "sub_id", "id") if c in cols), None)
    if id_col is None:
        raise KeyError("Could not find a subject id column in participants.tsv (tried participant_id/subject_id/sub_id/id)")

    mask = pd.Series(True, index=df.index)

    # Age
    if age_range is not None:
        if "age" not in cols:
            raise KeyError("Age filter requested but 'age' column not found in participants.tsv")
        a_min, a_max = age_range
        m = df["age"].between(a_min, a_max)
        if include_nan:
            m |= df["age"].isna()
        mask &= m

    # Sex
    if sex is not None:
        if "sex" not in cols:
            raise KeyError("Sex filter requested but 'sex' column not found in participants.tsv")
        wanted = {s.upper() for s in sex}
        sx = df["sex"].astype(str).str.upper()
        m = sx.isin(wanted)
        if include_nan:
            m |= df["sex"].isna()
        mask &= m

    # Handedness
    if handedness is not None:
        if "handedness" not in cols:
            raise KeyError("Handedness filter requested but 'handedness' column not found in participants.tsv")
        wanted = {h.upper() for h in handedness}
        hd = df["handedness"].astype(str).str.upper()
        m = hd.isin(wanted)
        if include_nan:
            m |= df["handedness"].isna()
        mask &= m

    # Numeric factor ranges
    if factor_ranges:
        for col, (lo, hi) in factor_ranges.items():
            c = col.lower()
            if c not in cols:
                raise KeyError(f"Factor '{col}' not found in participants.tsv")
            m = df[c].between(lo, hi)
            if include_nan:
                m |= df[c].isna()
            mask &= m

    subs = df.loc[mask, id_col].astype(str).tolist()
    subs = [s if s.startswith("sub-") else f"sub-{s}" for s in subs]
    if limit is not None:
        subs = subs[:limit]
    return subs


# ---------- 3) (Optional) helper: age binning for grouping ----------

def add_age_bin(
    participants_df: pd.DataFrame,
    *,
    bins: Tuple[float, ...] = (5, 10, 15, 21),
    labels: Tuple[str, ...] = ("5-10", "11-15", "16-21"),
    column_name: str = "age_bin"
) -> pd.DataFrame:
    """
    Return a copy of participants_df with an 'age_bin' categorical column for grouping.
    """
    df = participants_df.copy()
    if "age" not in df.columns:
        raise KeyError("'age' column not found to create bins")
    df[column_name] = pd.cut(df["age"], bins=bins, labels=labels, include_lowest=True, right=True)
    return df


In [7]:
data_root = "/mount/NAS-public-dataset/HBN-EEG"
release = 9

# 1) load & inspect (debug here freely)
p = get_participants_df(data_root, release)
print(p.head())
print(p.columns)

     participant_id release_number sex      age  ehq_total commercial_use  \
0  sub-NDARAC589YMB             R9   M  14.5916      93.38            Yes   
1  sub-NDARAC853CR6             R9   M   9.1899     -66.67            Yes   
2  sub-NDARAE710YWG             R9   M   9.9860      96.67            Yes   
3  sub-NDARAH239PGG             R9   M   7.7552     100.00            Yes   
4  sub-NDARAL897CYV             R9   M  12.9895      26.68            Yes   

  full_pheno  p_factor  attention  internalizing  ...  thepresent  \
0         No    -0.684     -0.383         -1.299  ...   available   
1        Yes     0.437      0.183         -1.518  ...   available   
2        Yes     1.239     -0.882         -0.583  ...   available   
3        Yes    -0.822     -0.813          1.276  ...   available   
4        Yes     0.382     -0.191         -0.959  ...   available   

  diaryofawimpykid contrastchangedetection_1 contrastchangedetection_2  \
0        available                 available    

In [8]:
# 2) filter, using the loaded df (no re-read)
subs_5_10 = filter_subjects(
    data_root, release,
    age_range=(5, 10),
    participants_df=p
)

# 3) add age bins for later grouping
p_binned = add_age_bin(p)
p_binned[["participant_id", "age", "age_bin"]].head()

,participant_id,age,age_bin
0,sub-NDARAC589YMB,14.5916,11-15
1,sub-NDARAC853CR6,9.1899,5-10
2,sub-NDARAE710YWG,9.9860,5-10
3,sub-NDARAH239PGG,7.7552,5-10
4,sub-NDARAL897CYV,12.9895,11-15
